# 🔊 Anomalous Sound Detection — Colab Training

This notebook runs the full training pipeline on Google Colab with GPU.

**Pipeline:** Clone repo → Install deps → Upload data → Preprocess → Train → Evaluate → Download artifacts

---

## 1. Setup Environment

In [ ]:
# Clone your repo
!git clone https://github.com/SARWAGYASHAH/Anomalous-Sound-Detection-using-Spectrograms.git
%cd Anomalous-Sound-Detection-using-Spectrograms

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -e . -q

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Upload & Extract Data

Upload `dev_data_gearbox.zip` to Colab, or mount Google Drive.

In [ ]:
# Option A: Upload directly (for small datasets)
# from google.colab import files
# uploaded = files.upload()  # Upload dev_data_gearbox.zip

# Option B: Mount Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

# Copy zip from Drive to project
!cp '/content/drive/MyDrive/dev_data_gearbox.zip' Data/dev_data_gearbox.zip

In [ ]:
# Extract dataset
import zipfile, os
with zipfile.ZipFile('Data/dev_data_gearbox.zip', 'r') as z:
    z.extractall('Data/')
print('Extracted!')
!ls Data/gearbox/

## 3. Run Preprocessing Pipeline

Converts raw `.wav` → mel spectrograms (`.npy`)

In [ ]:
!python pipeline/01_preprocess.py

In [ ]:
# Verify processed data
import numpy as np
from pathlib import Path

for d in ['train/normal', 'source_test/normal', 'source_test/anomaly']:
    p = Path(f'Data/processed/{d}')
    if p.exists():
        files = list(p.glob('*.npy'))
        print(f'{d}: {len(files)} files')

s = np.load(list(Path('Data/processed/train/normal').glob('*.npy'))[0])
print(f'Spectrogram shape: {s.shape}, range: [{s.min():.2f}, {s.max():.2f}]')

## 4. Load Config

In [ ]:
import yaml

with open('config/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

print(f'Seed: {config["seed"]}')
print(f'Model: {config["model"]["type"]}')
print(f'LR: {config["training"]["learning_rate"]}')
print(f'Epochs: {config["training"]["epochs"]}')
print(f'Batch: {config["training"]["batch_size"]}')

## 5. Set Seed for Reproducibility

In [ ]:
from src.utils.seed import set_seed
set_seed(config['seed'])

## 6. Create DataLoaders

In [ ]:
from src.data.dataset import AutoencoderDataset, SpectrogramDataset
from torch.utils.data import DataLoader, random_split

# Training data (normal only — autoencoder learns normal patterns)
full_dataset = AutoencoderDataset('Data/processed/train')
print(f'Total training samples: {len(full_dataset)}')

# Train/val split
train_split = config['data']['train_split']
train_size = int(len(full_dataset) * train_split)
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

# DataLoaders
batch_size = config['training']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Test data (with labels for evaluation)
test_dataset = SpectrogramDataset('Data/processed/source_test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
print(f'Test: {len(test_dataset)} (normal={test_dataset._count_label(0)}, anomaly={test_dataset._count_label(1)})')

## 7. Define Autoencoder Model

Frame-level FC autoencoder: each 128-dim mel frame is encoded → latent → decoded.
Anomaly score = mean reconstruction error across all time frames.

In [ ]:
import torch
import torch.nn as nn


class FrameAutoencoder(nn.Module):
    """
    Frame-level fully connected autoencoder.
    
    Input spectrogram (1, 128, T) is reshaped to (T, 128) frames,
    each frame is independently encoded and decoded,
    then reshaped back to (1, 128, T).
    """

    def __init__(self, input_dim=128, latent_dim=32,
                 encoder_layers=None, decoder_layers=None,
                 activation='relu', dropout=0.2):
        super().__init__()
        
        if encoder_layers is None:
            encoder_layers = [128, 64, 32]
        if decoder_layers is None:
            decoder_layers = [32, 64, 128]
        
        # Activation function
        act_fn = {
            'relu': nn.ReLU(),
            'leaky_relu': nn.LeakyReLU(0.2),
            'elu': nn.ELU(),
        }[activation]
        
        # Build encoder
        enc_modules = []
        prev_dim = input_dim
        for dim in encoder_layers:
            enc_modules.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                act_fn,
                nn.Dropout(dropout),
            ])
            prev_dim = dim
        enc_modules.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*enc_modules)
        
        # Build decoder
        dec_modules = []
        prev_dim = latent_dim
        for dim in decoder_layers:
            dec_modules.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                act_fn,
                nn.Dropout(dropout),
            ])
            prev_dim = dim
        dec_modules.append(nn.Linear(prev_dim, input_dim))
        dec_modules.append(nn.Sigmoid())  # Output in [0, 1] to match normalized spectrograms
        self.decoder = nn.Sequential(*dec_modules)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        """x shape: (batch, 1, n_mels, time) → same shape output."""
        b, c, n_mels, t = x.shape
        
        # Reshape: (B, 1, 128, T) → (B*T, 128)
        x_frames = x.squeeze(1).permute(0, 2, 1).reshape(-1, n_mels)
        
        # Encode → decode each frame
        z = self.encode(x_frames)
        x_hat = self.decode(z)
        
        # Reshape back: (B*T, 128) → (B, 1, 128, T)
        x_hat = x_hat.reshape(b, t, n_mels).permute(0, 2, 1).unsqueeze(1)
        
        return x_hat


# Build model from config
model_cfg = config['model']['autoencoder']
model = FrameAutoencoder(
    input_dim=model_cfg['input_dim'],
    latent_dim=model_cfg['latent_dim'],
    encoder_layers=model_cfg['encoder_layers'],
    decoder_layers=model_cfg['decoder_layers'],
    activation=model_cfg['activation'],
    dropout=model_cfg['dropout'],
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: FrameAutoencoder')
print(f'Parameters: {total_params:,}')
print(f'Device: {device}')
print(model)

## 8. Training Loop with MLflow

In [ ]:
import mlflow
import time
from tqdm import tqdm

# Training config
train_cfg = config['training']
epochs = train_cfg['epochs']
lr = train_cfg['learning_rate']
weight_decay = train_cfg['weight_decay']

# Loss & Optimizer
criterion = nn.MSELoss()

optimizer_map = {
    'adam': torch.optim.Adam,
    'adamw': torch.optim.AdamW,
    'sgd': torch.optim.SGD,
}
optimizer = optimizer_map[train_cfg['optimizer']](
    model.parameters(), lr=lr, weight_decay=weight_decay
)

# Scheduler
sched_cfg = train_cfg['scheduler']
if sched_cfg['type'] == 'step':
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=sched_cfg['step_size'], gamma=sched_cfg['gamma']
    )
elif sched_cfg['type'] == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
else:
    scheduler = None

print(f'Optimizer: {train_cfg["optimizer"]}, LR: {lr}, Epochs: {epochs}')

In [ ]:
# MLflow setup
mlflow_cfg = config['mlflow']
mlflow.set_tracking_uri(mlflow_cfg['tracking_uri'])
mlflow.set_experiment(mlflow_cfg['experiment_name'])

# Early stopping state
es_cfg = train_cfg['early_stopping']
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_params({
        'seed': config['seed'],
        'epochs': epochs,
        'batch_size': train_cfg['batch_size'],
        'learning_rate': lr,
        'optimizer': train_cfg['optimizer'],
        'weight_decay': weight_decay,
        'latent_dim': model_cfg['latent_dim'],
        'dropout': model_cfg['dropout'],
        'n_mels': config['spectrogram']['n_mels'],
        'n_fft': config['spectrogram']['n_fft'],
    })
    
    print(f'MLflow Run ID: {run.info.run_id}')
    print(f'Training started...\n')
    
    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        epoch_train_loss = 0.0
        
        for batch_input, batch_target in train_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)
            
            output = model(batch_input)
            loss = criterion(output, batch_target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
        
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # --- Validate ---
        model.eval()
        epoch_val_loss = 0.0
        
        with torch.no_grad():
            for batch_input, batch_target in val_loader:
                batch_input = batch_input.to(device)
                batch_target = batch_target.to(device)
                output = model(batch_input)
                loss = criterion(output, batch_target)
                epoch_val_loss += loss.item()
        
        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Step scheduler
        if scheduler is not None:
            scheduler.step()
        
        # Log to MLflow
        mlflow.log_metrics({
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'lr': optimizer.param_groups[0]['lr'],
        }, step=epoch)
        
        # Print progress
        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {avg_train_loss:.6f} | '
                  f'Val Loss: {avg_val_loss:.6f} | '
                  f'LR: {optimizer.param_groups[0]["lr"]:.6f}')
        
        # --- Early Stopping ---
        if es_cfg['enabled']:
            if avg_val_loss < best_val_loss - es_cfg['min_delta']:
                best_val_loss = avg_val_loss
                patience_counter = 0
                # Save best model
                torch.save(model.state_dict(), 'artifacts/models/best_model.pth')
            else:
                patience_counter += 1
                if patience_counter >= es_cfg['patience']:
                    print(f'\nEarly stopping at epoch {epoch} (patience={es_cfg["patience"]})')
                    break
    
    # Save final model
    torch.save(model.state_dict(), 'artifacts/models/final_model.pth')
    
    # Log model to MLflow
    if mlflow_cfg.get('log_models', True):
        mlflow.pytorch.log_model(model, 'model')
    
    print(f'\nTraining complete! Best val loss: {best_val_loss:.6f}')

## 9. Plot Training Curves

In [ ]:
from src.utils.visualization import plot_loss_curve

plot_loss_curve(
    train_losses, val_losses,
    title='Training vs Validation Loss',
    save_path='artifacts/logs/loss_curve.png',
    show=True,
)

## 10. Evaluate on Test Set

In [ ]:
import numpy as np
from src.utils.metrics import evaluate_all
from src.utils.visualization import plot_anomaly_scores, plot_roc_curve
from sklearn.metrics import roc_curve

# Load best model
model.load_state_dict(torch.load('artifacts/models/best_model.pth'))
model.eval()

# Compute reconstruction errors on test set
all_scores = []
all_labels = []

with torch.no_grad():
    for batch, labels in test_loader:
        batch = batch.to(device)
        output = model(batch)
        # Mean reconstruction error per sample
        errors = torch.mean((batch - output) ** 2, dim=[1, 2, 3])
        all_scores.extend(errors.cpu().numpy())
        all_labels.extend(labels.numpy())

scores = np.array(all_scores)
labels = np.array(all_labels)

print(f'Test samples: {len(scores)}')
print(f'Normal scores:  mean={scores[labels==0].mean():.6f}, std={scores[labels==0].std():.6f}')
print(f'Anomaly scores: mean={scores[labels==1].mean():.6f}, std={scores[labels==1].std():.6f}')

In [ ]:
# Full evaluation
results = evaluate_all(labels, scores)

print('\n=== Evaluation Results ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Log metrics to MLflow
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metrics({
        'test_auc_roc': results['auc_roc'],
        'test_pauc': results['pauc'],
        'test_f1': results['f1'],
        'test_precision': results['precision'],
        'test_recall': results['recall'],
    })

In [ ]:
# Plot anomaly score distribution
plot_anomaly_scores(
    scores, labels,
    threshold=results['threshold'],
    title='Anomaly Score Distribution',
    save_path='artifacts/logs/anomaly_scores.png',
    show=True,
)

# Plot ROC curve
fpr, tpr, _ = roc_curve(labels, scores)
plot_roc_curve(
    fpr, tpr, results['auc_roc'],
    title='ROC Curve',
    save_path='artifacts/logs/roc_curve.png',
    show=True,
)

## 11. Save Metadata & Artifacts

In [ ]:
from src.utils.metadata_tracker import MetadataTracker
from src.utils.artifact_versioner import ArtifactVersioner
from datetime import datetime

# Version the model
versioner = ArtifactVersioner('artifacts/models')
version_dir = versioner.resolve_version(config)
versioner.save_config_snapshot(version_dir, config)

# Copy best model to versioned dir
import shutil
shutil.copy('artifacts/models/best_model.pth', version_dir / 'model.pth')
print(f'Model saved to: {version_dir}')

# Save metadata
tracker = MetadataTracker('artifacts/metadata')
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
tracker.save(
    run_id=run_id,
    params={
        'seed': config['seed'],
        'epochs': len(train_losses),
        'batch_size': train_cfg['batch_size'],
        'learning_rate': lr,
        'optimizer': train_cfg['optimizer'],
        'latent_dim': model_cfg['latent_dim'],
    },
    metrics=results,
    artifacts={
        'model_path': str(version_dir / 'model.pth'),
        'config_snapshot': str(version_dir / 'config_snapshot.yaml'),
    },
    mlflow_run_id=run.info.run_id,
    experiment_name=mlflow_cfg['experiment_name'],
)
print(f'Metadata saved for run: {run_id}')

## 12. Download Artifacts to Local Machine

In [ ]:
# Zip all artifacts for download
!zip -r artifacts_export.zip artifacts/

from google.colab import files
files.download('artifacts_export.zip')
print('Download started! Extract into your local project root.')